# Lab 5.2. Fine-tuning BERT с помощью Hugging Face Trainer

В этой лабораторной мы решим задачу классификации отзывов Yelp по числу звёзд — от 1 до 5. Все этапы обучения выполняются через высокоуровневые модули библиотеки `transformers`, без написания собственного training loop и без прямого использования `torch`.

Мы не только обучим модель, но и:

1. сравним качество до и после fine-tuning;
2. построим confusion matrix и найдём самые частые ошибки;
3. исследуем примеры, в которых модель не уверена;
4. запустим модель через `pipeline` на собственных текстах.


## 1. Подготовка окружения

`Trainer` берёт на себя batching, backpropagation, обновление параметров, evaluation и логирование. Для его работы также требуется библиотека `accelerate`.


In [1]:
%%capture
!pip install -q transformers datasets evaluate accelerate scikit-learn seaborn


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate

from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

set_seed(42)


## 2. Датасет Yelp Review Full

Каждый отзыв имеет метку от `0` до `4`, соответствующую оценкам от одной до пяти звёзд. Полный датасет достаточно большой, поэтому для семинара используем небольшие воспроизводимые подвыборки.


In [4]:
dataset = load_dataset("Yelp/yelp_review_full")

print(dataset)
print("Метка первого отзыва:", dataset["train"][0]["label"])
print("Текст первого отзыва:")
print(dataset["train"][0]["text"][:500])


README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})
Метка первого отзыва: 4
Текст первого отзыва:
dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about hi


In [5]:
train_size = 2_000
eval_size = 1_000

small_train_dataset = dataset["train"].shuffle(seed=42).select(range(train_size))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(eval_size))

label_counts = pd.Series(small_train_dataset["label"]).value_counts().sort_index()
display(pd.DataFrame({
    "звёзды": label_counts.index + 1,
    "число отзывов": label_counts.values,
}))


,звёзды,число отзывов
0,1,438
1,2,404
2,3,370
3,4,406
4,5,382


## 3. Tokenization и dynamic padding

Ограничим длину последовательности до 128 tokens, чтобы ускорить занятие. Мы не добавляем padding при tokenization: `DataCollatorWithPadding` дополнит каждый batch только до длины самого длинного примера в нём. Обычно это эффективнее, чем дополнять весь датасет до фиксированной максимальной длины.


In [6]:
model_name = "google-bert/bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)


tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_eval = small_eval_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Число tokens в первом примере:", len(tokenized_train[0]["input_ids"]))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Число tokens в первом примере: 75


## 4. Модель и метрики

Загружаем предварительно обученный BERT и добавляем новую classification head для пяти классов. Вес этой head сначала случаен, поэтому до fine-tuning качество должно быть близко к случайному baseline — около 20% accuracy.

Кроме accuracy используем **macro F1**: эта метрика отдельно вычисляет F1 для каждого класса, а затем усредняет результаты.


In [7]:
id2label = {i: f"{i + 1} star" for i in range(5)}
label2id = {name: idx for idx, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions, references=labels
        )["accuracy"],
        "macro_f1": f1_metric.compute(
            predictions=predictions, references=labels, average="macro"
        )["f1"],
    }


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
training_args = TrainingArguments(
    output_dir="results_bert_yelp",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 5. Качество до и после fine-tuning

Сначала оценим BERT со случайно инициализированной classification head. Затем `Trainer` выполнит весь training loop и мы повторим evaluation на тех же данных.


In [11]:
metrics_before = trainer.evaluate()
print("До fine-tuning:")
print({key: round(value, 4) for key, value in metrics_before.items() if key.startswith("eval_")})


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
No log,1.644188,0,0.184000,0.081973


До fine-tuning:
{'eval_loss': 1.6442, 'eval_accuracy': 0.184, 'eval_macro_f1': 0.082}


In [ ]:
train_result = trainer.train()
metrics_after = trainer.evaluate()

comparison = pd.DataFrame(
    {
        "до fine-tuning": [
            metrics_before["eval_accuracy"],
            metrics_before["eval_macro_f1"],
        ],
        "после fine-tuning": [
            metrics_after["eval_accuracy"],
            metrics_after["eval_macro_f1"],
        ],
    },
    index=["accuracy", "macro F1"],
)
display(comparison.round(4))


Epoch,Training Loss,Validation Loss


## 6. Какие классы модель путает?

Одной accuracy недостаточно для понимания поведения модели. Получим предсказания на evaluation set и построим confusion matrix. Строки соответствуют истинной оценке, столбцы — предсказанной.


In [ ]:
prediction_output = trainer.predict(tokenized_eval)
logits = prediction_output.predictions
true_labels = prediction_output.label_ids
predicted_labels = np.argmax(logits, axis=-1)

print(classification_report(
    true_labels,
    predicted_labels,
    labels=list(range(5)),
    target_names=[f"{i} star" for i in range(1, 6)],
    digits=3,
    zero_division=0,
))

matrix = confusion_matrix(true_labels, predicted_labels, labels=list(range(5)))
plt.figure(figsize=(7, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(1, 6),
    yticklabels=range(1, 6),
)
plt.xlabel("Предсказанное число звёзд")
plt.ylabel("Истинное число звёзд")
plt.title("Confusion matrix для отзывов Yelp")
plt.tight_layout()
plt.show()


### 6.1 Анализ уверенности и ошибок

Преобразуем logits в вероятности с помощью softmax, реализованного средствами NumPy. Разница между двумя наибольшими вероятностями — простой показатель уверенности: чем она меньше, тем сложнее модели выбрать класс.


In [ ]:
shifted_logits = logits - logits.max(axis=1, keepdims=True)
probabilities = np.exp(shifted_logits) / np.exp(shifted_logits).sum(axis=1, keepdims=True)
sorted_probabilities = np.sort(probabilities, axis=1)
confidence_margin = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]

analysis = pd.DataFrame({
    "text": small_eval_dataset["text"],
    "true_stars": true_labels + 1,
    "predicted_stars": predicted_labels + 1,
    "confidence": probabilities.max(axis=1),
    "margin": confidence_margin,
})

errors = analysis[analysis["true_stars"] != analysis["predicted_stars"]]
display(
    errors.sort_values("margin")
    .head(10)
    .assign(text=lambda frame: frame["text"].str.slice(0, 250))
)


Обратите внимание: соседние оценки, например 3 и 4 звезды, семантически близки и часто смешиваются даже для человека. Полезно обсудить, всегда ли ошибка на одну звезду так же серьёзна, как ошибка на четыре звезды. Для ordinal classification можно использовать метрики, учитывающие расстояние между классами.


In [ ]:
mean_absolute_star_error = np.abs(true_labels - predicted_labels).mean()
large_error_share = (np.abs(true_labels - predicted_labels) >= 2).mean()

print(f"Средняя абсолютная ошибка: {mean_absolute_star_error:.3f} звезды")
print(f"Доля ошибок на 2+ звезды: {large_error_share:.2%}")


## 7. Inference через pipeline

`pipeline` скрывает tokenization, запуск модели и преобразование logits в scores. Передадим несколько собственных отзывов и посмотрим распределение вероятностей по всем пяти классам.


In [ ]:
classifier = pipeline(
    task="text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    top_k=None,
)

new_reviews = [
    "The food was outstanding and the staff made our evening perfect.",
    "The meal was fine, but the service was slow and the room was noisy.",
    "I waited for an hour and the food arrived cold. Never again.",
]

for review, scores in zip(new_reviews, classifier(new_reviews, truncation=True, max_length=128)):
    ordered_scores = sorted(scores, key=lambda item: item["score"], reverse=True)
    print(f"\n{review}")
    for item in ordered_scores:
        print(f"  {item['label']:>6}: {item['score']:.3f}")
